In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [3]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

#Creating the entangled qbit to distribute to Bob and Alice
def entangled_bits():
    circuit = QuantumCircuit(2,2)
    circuit.h(0)
    circuit.cx(0,1)
    circuit.z(1)
    circuit.x(0)
    return circuit


# Creating a qbit to generate a random number between 1 and 3
def quantum_random_generator():
    random_qbit = QuantumCircuit(2,2)

    t_matrix = [ [ (1/math.sqrt(3)) , (math.sqrt(2)/math.sqrt(3))],
          [ ((-1) * (math.sqrt(2)/math.sqrt(3))) , (1/math.sqrt(3))] ]
    
    t_operator = Operator(t_matrix)
    
    random_qbit.unitary(t_operator, [0])
    random_qbit.h(1)
    
    random_qbit.measure(range(2), range(2))
        
    backend = BasicSimulator()
    qc_compiled = transpile(random_qbit, backend)
    job_sim = backend.run(qc_compiled, shots = 1)
    result_sim = job_sim.result()
    result = result_sim.get_counts(qc_compiled)
    
    result = list(result)[0]
    
    if result[1] == '0':
        return 1
    elif result[0] == '1':
        return 2
    else:
        return 3

# Function to simulate an attacker, measuring the shared qbit
def attacker(c):
    c.measure(range(2), range(2))
    
    backend = BasicSimulator()
    qc_compiled = transpile(c, backend)
    job_sim = backend.run(qc_compiled, shots = 1)
    result_sim = job_sim.result()
    result = result_sim.get_counts(qc_compiled)

# Function to execute the Ekert protocol
def execute_protocol(n):

    # Operator matrices for the w and v states 
    root2 = math.sqrt(2)
    denom1 = math.sqrt(4 + 2*root2)
    denom2 = math.sqrt(4 - 2*root2) 
    
    w_operator_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                            [  1 / denom2 , (root2 - 1) / denom2 ] ]
    w_operator = Operator(w_operator_matrix) 
    
    
    v_operator_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                            [ -1 / denom2 , (root2 - 1) / denom2 ] ]
    v_operator = Operator(v_operator_matrix)
    

    # Converter and inverter
    converter = {'0': 1, '1': -1}
    inverter = {'0': '1', '1': '0'}
    
    final_key = ""

    # total value of each pairing of unitary operation and number of times each pairing has occurred, for averaging
    s_value = {"xw": 0, "xv": 0, "zw": 0, "zv": 0}
    s_times = {"xw": 0, "xv": 0, "zw": 0, "zv": 0}
    
    for i in range(0, int((9*n)/2)):

        # Create the entangled bit
        circuit = entangled_bits()

        # Attacker eavesdrops, measuring the qbit
        attacker(circuit)
        
        # Alice chooses her operator
        alice_operator_num = quantum_random_generator()
        
        # Bob chooses his operator
        bob_operator_num = quantum_random_generator()

        # Alice applies the appropriate unitary operator
        if alice_operator_num == 1:
            circuit.h(0)
        elif alice_operator_num == 2:
            circuit.unitary(w_operator, [0])
        else:
            pass    

        # Bob applies the appropriate unitary operator
        if bob_operator_num == 1:
            circuit.unitary(w_operator, [1])
        elif bob_operator_num == 2:
            pass
        else:
            circuit.unitary(v_operator, [1])
            
        # Both bits are measured
        circuit.measure(range(2), range(2))
        
        backend = BasicSimulator()
        qc_compiled = transpile(circuit, backend)
        job_sim = backend.run(qc_compiled, shots = 1)
        result_sim = job_sim.result()
        result = result_sim.get_counts(qc_compiled)
    
        result = list(result)[0]
        alice_bit = converter[result[1]]
        bob_bit = converter[inverter[result[0]]]
    
        #Bob and Alice compare operator choices and either add to the shared key, or add to the counts for averaging for s calculation
        if alice_operator_num == 1 and bob_operator_num == 1:
            s_times["xw"] += 1
            s_value["xw"] += (alice_bit * bob_bit)
        elif alice_operator_num == 1 and bob_operator_num == 3:
            s_times["xv"] += 1
            s_value["xv"] += (alice_bit * bob_bit)
        elif alice_operator_num == 3 and bob_operator_num == 1:
            s_times["zw"] += 1
            s_value["zw"] += (alice_bit * bob_bit)
        elif alice_operator_num == 3 and bob_operator_num == 3:
            s_times["zv"] += 1
            s_value["zv"] += (alice_bit * bob_bit)
        elif alice_operator_num == 2 and bob_operator_num == 1:
            final_key += result[0] 
        elif alice_operator_num == 3 and bob_operator_num == 2:
            final_key += result[0] 
        else:
            pass

    # Calculates s, with contingency for a division by 0 error
    for k in s_value.keys():
        if s_times[k] == 0:
            s_value[k] = 0
        else:    
            s_value[k] = (s_value[k]/s_times[k])
    
    s = abs(s_value["xw"] - s_value["xv"] + s_value["zw"] + s_value["zv"])
    
    print("Final key is: " + final_key)
    print("S value is: " + str(s))

#Execute protocol for a key of around length n
execute_protocol(500)




Final key is: 11000011000101111101001010001110000000101000000101111101100000010001000010000100000010001010010101011110101101100101110010011010110001100100000001001010010111100001100100100100010111000000001101101010100010011100000110111111001011100110111010100001101011100001010000110011100010001100011111011011000001001001000110110000110001101011010100000110100101010011001010000110111010010001110110010111011110101111001011010111110010100000100000001001110001110100010111100100001110101011001000101000101000000001001010
S value is: 1.263260213435668
